In [ ]:
import math
import random
from collections import Counter

import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

from peft import LoraConfig, get_peft_model, PeftModel, PeftConfig

/home/amily/assignment-3-evelynh037/assignment-3-evelynh037-1/myenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# load data

In [2]:
# load data and train split
file_path = "data/narrativeqa_movies_lora_train_1000.jsonl"

df = pd.read_json(file_path, lines=True)

train_df, test_df = train_test_split(
    df,
    test_size=0.1,
    random_state=42,
    shuffle=True
)


# load model

In [ ]:
# Make sure GPU is available
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-1.7B-Instruct")

# Load model on GPU safely
model = AutoModelForCausalLM.from_pretrained(
    "HuggingFaceTB/SmolLM2-1.7B-Instruct",
    torch_dtype=torch.float16,  # reduces memory usage
    device_map="auto"           # automatically places layers on GPU
)

# Prepare prompt
messages = [{"role": "user", "content": "Who are you?"}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
).to(device)  # send input tensors to GPU

# Generate output
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=40)

# Decode only generated tokens
generated_text = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
print(generated_text)

Using device: cuda


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 218/218 [00:00<00:00, 233.75it/s, Materializing param=model.norm.weight]                              


I am SmolLM, an AI model trained by Hugging Face. I am designed to provide information and answer questions to the best of my ability. I am here to assist you with your queries


# helper function 

In [ ]:
def generate_answer_from_prompt(prompt, model, max_new_tokens: int = 64) -> str:
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)

    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    generated_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    return generated_text.strip()


def build_prompt(row):
    """Format the prompt so the model gives a short, direct answer.

    This reduces the chance the model invents options like A/B/C/D.
    """
    return (
        row["prompt"]
        + "\n\nPlease answer the question in one short sentence. "
        + "Do not list options, only give the direct answer.\n\nAnswer:"
    )

def token_f1(prediction: str, reference: str) -> float:
    if not isinstance(prediction, str) or not isinstance(reference, str):
        return 0.0

    pred_tokens = prediction.lower().split()
    ref_tokens = reference.lower().split()

    if not pred_tokens or not ref_tokens:
        return 0.0

    pred_counts = Counter(pred_tokens)
    ref_counts = Counter(ref_tokens)

    common = sum((pred_counts & ref_counts).values())
    if common == 0:
        return 0.0

    precision = common / len(pred_tokens)
    recall = common / len(ref_tokens)
    if precision + recall == 0:
        return 0.0

    return 2 * precision * recall / (precision + recall)



# Base model

- get answer
- evaluation

In [ ]:
# get answer with base model
num_rows = len(test_df)
print(f"Generating predictions for {num_rows} test examples...")

predictions = []
for _, row in tqdm(test_df.iterrows(), total=num_rows):
    prompt = build_prompt(row)
    prediction = generate_answer_from_prompt(prompt, model)
    predictions.append(prediction)

test_df["prediction"] = predictions
test_df.head()
test_df.to_csv("data/test_df.csv")

Generating predictions for 100 test examples...


100%|██████████| 100/100 [01:16<00:00,  1.32it/s]


,prompt,response,question,prediction,token_f1,prediction_finetuned,token_f1_finetuned
521,Summary (context):\n Trevor Gooden (Dean Winte...,Trevor,Who is the prime suspent in a murder case?,Trevor Gooden is the prime suspect in a murder...,0.181818,Trevor Gooden is the prime suspect in a murder...,0.181818
737,"Summary (context):\n Harry (Danny Glover), an ...",they were delighted to see him,What reaction did suzie and gideon give after ...,Suzie and Gideon were delighted to see Harry a...,0.342857,Suzie and Gideon were delighted to see Harry a...,0.285714
740,Summary (context):\n After breaking out of jai...,Because Moco put Azul in jail in Mexico.,Why does Azul want vengeance against Moco?,Azul wants vengeance against Moco because Moco...,0.347826,Azul wants vengeance against Moco because Moco...,0.300000
660,"Summary (context):\n ""Zerophilia"" is a fiction...",By having sex with someone else who is also a ...,How can a zerophiliac become a-morphic?,A zerophiliac can become a-morphic by having s...,0.451613,A zerophiliac can become a-morphic by having s...,0.545455
411,Summary (context):\n Theater director Caden Co...,MacArthur Fellowship,What award does Caden win for the Death of a S...,Caden Cotard wins a MacArthur Fellowship for h...,0.105263,Caden wins a MacArthur Fellowship for his prod...,0.266667


In [ ]:
# evaluation on base model
num_rows = len(test_df)
print(f"Computing token-level F1 for {num_rows} test examples...")

f1_scores = []
for _, row in tqdm(test_df.iterrows(), total=num_rows):
    reference = row["response"]
    prediction = row["prediction"]
    f1_scores.append(token_f1(prediction, reference))

# Store per-example F1 and report the average
test_df["token_f1"] = f1_scores

avg_f1 = np.mean(test_df["token_f1"])
print(f"Average token F1 over test set: {avg_f1:.3f}")

Computing token-level F1 for 100 test examples...


100%|██████████| 100/100 [00:00<00:00, 12431.99it/s]

Average token F1 over test set: 0.273


# LoRA Fine tunning

- data prep
- LoRA fine tunning
- evaluation

In [ ]:
# Build a Hugging Face Dataset from the train split
train_dataset = Dataset.from_pandas(train_df[["prompt", "response"]].reset_index(drop=True))

def format_example(example):
    messages = [
        {"role": "user", "content": example["prompt"]},
        {"role": "assistant", "content": example["response"]},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

formatted_train_dataset = train_dataset.map(format_example)

max_length = 512


def tokenize_function(example):
    tokenized = tokenizer(
        example["text"],
        truncation=True,
        max_length=max_length,
        padding="max_length",
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized


# Tokenize the dataset and keep only the model inputs/labels
tokenized_train_dataset = formatted_train_dataset.map(
    tokenize_function, remove_columns=formatted_train_dataset.column_names
)

len(tokenized_train_dataset)

Map: 100%|██████████| 900/900 [00:01<00:00, 577.10 examples/s]


900

In [ ]:
# Configure LoRA adapters
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"],
)

# Wrap the base model with LoRA
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 1,572,864 || all params: 1,712,949,248 || trainable%: 0.0918


In [ ]:
# load base model
base_model_name = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(base_model_name)
base_model = AutoModelForCausalLM.from_pretrained(base_model_name)

# setup
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

# wrap base model
model_lora = get_peft_model(base_model, lora_config)
model_lora.print_trainable_parameters()

# load data
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# setup
training_args = TrainingArguments(
    output_dir="smollm2-lora-narrativeqa",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    logging_steps=20,
    save_strategy="no", 
    fp16=torch.cuda.is_available(),
    remove_unused_columns=False,
)

# trainer
trainer = Trainer(
    model=model_lora,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    data_collator=data_collator,
)

trainer.train()

# save model
torch.save(model_lora.state_dict(), "../smollm2-lora-narrativeqa.pt")
model_lora.save_pretrained("smollm2-lora-narrativeqa")
tokenizer.save_pretrained("smollm2-lora-narrativeqa-tokenizer")


# Load base model
base_model = AutoModelForCausalLM.from_pretrained(base_model_name)

# Load LoRA fine-tuned model
model_finetuned = PeftModel.from_pretrained(base_model, "smollm2-lora-narrativeqa")

# Move models to GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
base_model.to(device)
model_finetuned.to(device)

print(f"Base model device: {next(base_model.parameters()).device}")
print(f"Fine-tuned model device: {next(model_finetuned.parameters()).device}")
def generate_answer_from_prompt(prompt, model=model_finetuned, tokenizer=tokenizer, max_new_tokens=128):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)
    return tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

# example

prompt = "Context: Alice was in Wonderland. Question: Where did Alice go?"
answer = generate_answer_from_prompt(prompt)
print("Answer:", answer)


Loading weights: 100%|██████████| 218/218 [00:00<00:00, 655.55it/s, Materializing param=model.norm.weight]                              
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


trainable params: 3,145,728 || all params: 1,714,522,112 || trainable%: 0.1835


Step,Training Loss
20,2.819412
40,2.813986
60,2.788881
80,2.679162
100,2.611782
120,2.536344
140,2.507643
160,2.478828
180,2.465689
200,2.441279


Loading weights: 100%|██████████| 218/218 [00:00<00:00, 581.28it/s, Materializing param=model.norm.weight]                              


Answer:  Options:  A. to the Queen of Hearts  B. to the Red Queen  C. to the White Queen  D. to the Red Queen's castle  E. to the Queen of Hearts' castle
Answer:


In [ ]:
# Evaluate on test_set
preds = []
f1s = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    prompt = build_prompt(row)
    prediction = generate_answer_from_prompt(prompt, model_finetuned)
    preds.append(prediction)
    f1s.append(token_f1(prediction, row["response"]))

# Store predictions and F1 in new columns
test_df["prediction_finetuned"] = preds
test_df["token_f1_finetuned"] = f1s

# Print average token-level F1 for the test set
print("Average token F1 on full test set (fine-tuned model):",
      test_df["token_f1_finetuned"].mean())

# Inspect first few rows
test_df[["prompt", "response", "prediction_finetuned",
         "token_f1_finetuned"]].head()

test_df.to_csv("data/finetuned_test_df.csv")

100%|██████████| 100/100 [01:17<00:00,  1.29it/s]

Average token F1 on full test set (fine-tuned model): 0.3466037236405146


,prompt,response,prediction_finetuned,token_f1_finetuned
521,Summary (context):\n Trevor Gooden (Dean Winte...,Trevor,Trevor Gooden is the prime suspect in a murder...,0.181818
737,"Summary (context):\n Harry (Danny Glover), an ...",they were delighted to see him,Suzie and Gideon were delighted to see Harry a...,0.285714
740,Summary (context):\n After breaking out of jai...,Because Moco put Azul in jail in Mexico.,Azul wants vengeance against Moco because Moco...,0.300000
660,"Summary (context):\n ""Zerophilia"" is a fiction...",By having sex with someone else who is also a ...,A zerophiliac can become a-morphic by having s...,0.545455
411,Summary (context):\n Theater director Caden Co...,MacArthur Fellowship,Caden wins a MacArthur Fellowship for his prod...,0.266667


# Comparison and comments

In [44]:
base = pd.read_csv("data/test_df.csv")
lora = pd.read_csv("data/finetuned_test_df.csv")

# merge two dataframes
merged_df = pd.concat([base, lora])
improved = merged_df[merged_df["token_f1_finetuned"] > merged_df["token_f1"]]
improved[["response", "prediction", "prediction_finetuned"]]

,response,prediction,prediction_finetuned
3,By having sex with someone else who is also a ...,A zerophiliac can become a-morphic by having s...,A zerophiliac can become a-morphic by having s...
4,MacArthur Fellowship,Caden Cotard wins a MacArthur Fellowship for h...,Caden wins a MacArthur Fellowship for his prod...
6,Betsy and Joan,The names of the Blanding daughters are Betsy ...,Betsy and Joan
9,neighbors have grown suspicious because they d...,Filmore and Marie-Noel move from their ranch b...,They move from their ranch because their neigh...
10,"Portland, Maine","The crocodile is taken to Portland, Maine.","Portland, Maine."
...,...,...,...
88,Luke continues to experience partial transform...,After Luke meets Michelle and begins going out...,Luke meets Michelle and begins going out with ...
90,Sera,Ben Sanderson (Cage) almost hits Sera (Shue) o...,Sera is the one who almost hits Ben Sanderson.
92,To ruin his sister's relationship with her boy...,Falco's boss wants him to break up the romance...,Falco's boss wants him to break up the romance...
93,Sarah,Kelly's love interest is Sarah.,Sarah is Kelly's love interest.


In [40]:
# Compute mean token-level F1 for base and fine-tuned models
base_mean_f1 = merged_df["token_f1"].mean()
finetuned_mean_f1 = merged_df["token_f1_finetuned"].mean()

# Create a summary dataframe
summary_df = {
    "Model": ["Base", "Fine-tuned"],
    "Mean_Token_F1": [base_mean_f1, finetuned_mean_f1]
}

summary_df = pd.DataFrame(summary_df)

print(summary_df)


        Model  Mean_Token_F1
0        Base       0.273229
1  Fine-tuned       0.346604


After fine-tunning, the model improve 7%, with the fine-tunned model, the model able to output better answers for 86 questions. 

The fine-tuned model is noticeably better at questions where the gold answer is a short phrase or named entity (e.g., a character name, location, or specific object). In many of the 86 improved cases it now selects the exact target phrase instead of paraphrasing or giving a related but incorrect span.

like The crocodile is taken to Portland, Maine.(base answer) and Portland, Maine. (fine-tunned answer)